> 📅 __Date: 2026-08-19__

# 🤖 **Decoding Strategies for Language Models**

> **Decoding strategy = the method used to convert a language model's next-token scores/probabilities into an actual generated sequence of tokens.**

For a decoder-only language model, text generation is **autoregressive**:

```text
Prompt
  ↓
Decoder-only Transformer
  ↓
Next-token logits
  ↓
Probability distribution
  ↓
Decoding strategy
  ↓
Choose next token
  ↓
Append token to context
  ↓
Run model again
  ↓
Repeat
```

The model itself learns a distribution over possible next tokens. **Decoding is the inference-time procedure that decides which token to actually emit.**

---

# 🏗️ **Decoder-Only Architecture**

<div align="center">

<img src="assets/Decoder_only_Arch.png" width="800" alt="Decoder-only Transformer architecture">

<p><em>Figure: Decoder-only Transformer architecture used for autoregressive text generation.</em></p>

</div>

A decoder-only language model generates one token at a time.

For a token sequence

```text
x₁, x₂, x₃, ..., xₜ₋₁
```

the model predicts the probability distribution of the next token \(x_t\).

$$
P(x_t \mid x_1, x_2, \ldots, x_{t-1})
$$

After choosing \(x_t\), the model uses the expanded context to predict \(x_{t+1}\).

```text
"The cat"
     ↓
predict next token
     ↓
"sits"
     ↓
"The cat sits"
     ↓
predict next token
     ↓
"on"
     ↓
"The cat sits on"
     ↓
...
```

---

# 🧠 **We Are Working with a Vocabulary**

**Suppose the model vocabulary is:**

```text
Vocab = [1st token, 2nd token, 3rd token, ..., Nth token]
```

The decoder produces a score for every vocabulary token.

**Conceptually:**

```text
Vocabulary
    ↓
┌─────────────────────────────┐
│ token 1   → score           │
│ token 2   → score           │
│ token 3   → score           │
│ token 4   → score           │
│ ...                         │
│ token N   → score           │
└─────────────────────────────┘
```

The raw scores are called **logits**.

**After converting logits into probabilities, we get:**

```text
token 1 → probability
token 2 → probability
token 3 → probability
...
token N → probability
```

**The probability for token \(v\) is:**

$$
P(x_t=v \mid x_{<t})
$$

**where:**

$$
x_{<t} = (x_1,x_2,\ldots,x_{t-1})
$$

---

# 🎯 **Output of the Decoder**

**Suppose the current context is:**

```text
"The"
```

**The model might produce a distribution such as:**

```text
dog      → 0.40
nice     → 0.25
cat      → 0.15
world    → 0.08
car      → 0.05
...
```

**The probabilities satisfy:**

$$
\sum_{v \in V} P(x_t=v \mid x_{<t}) = 1
$$

The model does **not** directly output the final word/token. The inference pipeline is:

```text
Decoder
  ↓
Logits
  ↓
Softmax / probability conversion
  ↓
Decoding strategy
  ↓
Selected token
```

---

# 🏆 **Token with Maximum Probability Will Be Chosen as the Next Token**

**The simplest decoding idea is:**

> **Always choose the token with the highest probability.**

For example:

```text
dog      → 0.40  ← choose
nice     → 0.25
cat      → 0.15
world    → 0.08
car      → 0.05
```

**Mathematically:**

$$
x_t
=
\underset{v \in V}{\operatorname{argmax}}
\; P(v \mid x_{<t})
$$

This is called **Greedy Search**.

---

# ⚠️ **Problems with Always Choosing the Maximum Probability**

If the prompt and all generation settings are the same, greedy decoding usually follows the same path.

```text
Same prompt
    ↓
Same distribution
    ↓
Same maximum-probability token
    ↓
Same next state
    ↓
Same next token
    ↓
Same output
```

**Therefore:**

1. **Generation is deterministic** under the same conditions.
2. **Repeated runs usually produce the same output.**
3. There is very little variation.
4. The output may feel more predictable or robotic.
5. The token that is best **right now** may not lead to the best **overall sequence**.

> **Important:** Deterministic does not mean bad. It can be useful when reproducibility and predictability are important.

---

# 📈 **Beam Search vs Human Choice**

<div align="center">

<img src="assets/beam_vs_human.png" width="800" alt="Beam search versus human sequence selection">

<p><em>Figure: A sequence-selection comparison over time.</em></p>

</div>

**The intuition is:**

```text
Greedy:
Choose the best-looking token now

Human / long-horizon reasoning:
Consider how the whole sequence may develop
```

This motivates methods that consider more than one local choice.

---

# 🔄 **Decoding Strategies**

**The main decoding strategies and generation controls are:**

```text
1. Greedy Search
2. Beam Search
3. Sampling
4. Top-K Sampling
5. Top-P / Nucleus Sampling
6. Temperature
7. Repetition Constraints
8. Length / Stopping Controls
```

These methods can be used independently or, in many practical generation setups, combined.

---

# 🤗 **Hugging Face Setup**

```python
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Once upon a time"

input_ids = tokenizer(prompt, return_tensors="pt")

gen_tokens = model.generate(**input_ids)

print(gen_tokens)
```

**Example token IDs:**

```text
tensor([[7454, 2402, 257, 640, 11, 262, 995, 373, 257, ...]])
```

**Decode token IDs back to text:**

```python
print(tokenizer.decode(gen_tokens[0]))
```

**Possible style of output:**

```text
"Once upon a time, the world was a place of great beauty and great danger. ..."
```

> **Note:** Exact generated text is model- and configuration-dependent. The same prompt can produce different text when stochastic sampling is enabled.

---

# 📏 **`max_new_tokens`**

## **`max_new_tokens` = How many new tokens to generate**

```python
gen_tokens = model.generate(
    **input_ids,
    max_new_tokens=100
)
```

If the prompt contains some number of input tokens, `max_new_tokens=100` means the model can generate **up to 100 additional tokens**.

```text
Input tokens
     +
New tokens
     =
Final sequence
```

### **Why is `max_new_tokens` useful?**

It directly specifies the number of **new tokens** to generate.

```python
gen_tokens = model.generate(
    **input_ids,
    max_new_tokens=100
)

print(tokenizer.decode(gen_tokens[0]))
```

---

# 🔁 **`no_repeat_ngram_size`**

## **`no_repeat_ngram_size` = Prevent repeated n-grams**

An **n-gram** is a sequence of \(n\) consecutive tokens.

**Suppose the generated text contains:**

```text
the world was
world was a
was a place
a place of
place of great
of great danger
```

**These are **trigrams** because:**

```text
n = 3
```

**For:**

```python
no_repeat_ngram_size=3
```

the generation process prevents an exact 3-token sequence from being generated again.

**Example:**

```python
gen_tokens = model.generate(
    **input_ids,
    max_new_tokens=100,
    no_repeat_ngram_size=3
)

print(tokenizer.decode(gen_tokens[0]))
```

**Conceptually:**

```text
Previously generated:
"the world was"

Later:
"the world was"  ← blocked
```

This can help reduce repetitive loops.

> **Important:** `no_repeat_ngram_size` is a repetition constraint, not a standalone decoding strategy.

---

# 🏆 **Greedy Search**

## **Greedy Search = Choose the token with maximum probability**

**At each time step:**

$$
x_t
=
\underset{v \in V}{\operatorname{argmax}}
\; P(v \mid x_{<t})
$$

<div align="center">

<img src="assets/greedy_search.png" width="800" alt="Greedy search decoding">

<p><em>Figure: Greedy search selects the highest-probability token at each generation step.</em></p>

</div>

**Example:**

```text
The →

dog      → 0.50  ← choose
nice     → 0.25
cat      → 0.15
car      → 0.10
```

**The result is:**

```text
The dog
```

**Then the model predicts again:**

```text
The dog →

has      → 0.50  ← choose
is       → 0.20
runs     → 0.15
...
```

**So:**

```text
The
 ↓
The dog
 ↓
The dog has
 ↓
The dog has ...
```

---

# 🎯 **Local Best Choice Does Not Always Mean Best Sequence**

**Consider:**

```text
The →
nice → 0.50
dog  → 0.40
```

**If we choose `nice`:**

```text
The nice
```

**and later:**

```text
P(woman | The nice) = 0.40
```

**Then the sequence probability is:**

$$
P(\text{nice},\text{woman}\mid\text{The})
=
0.50 \times 0.40
=
0.20
$$

**Now consider:**

```text
The dog
```

**with:**

```text
P(dog | The) = 0.40
```

**and later:**

```text
P(has | The dog) = 0.90
```

**Then:**

$$
P(\text{dog},\text{has}\mid\text{The})
=
0.40 \times 0.90
=
0.36
$$

**Therefore:**

```text
The → nice → woman
0.50 × 0.40 = 0.20

The → dog → has
0.40 × 0.90 = 0.36
```

The lower-probability first token can lead to a better overall sequence.

> **Greedy search optimizes the next decision locally, not the entire sequence globally.**

---

# 🌳 **Beam Search**

## **Beam Search = Keep several promising candidate sequences**

Greedy search keeps only one active sequence.

Beam search keeps multiple.

**The number of active candidates is controlled by:**

```python
num_beams
```

<div align="center">

<img src="assets/beam_search.png" width="800" alt="Beam search decoding">

<p><em>Figure: Beam search keeps multiple high-scoring candidate sequences and expands them over successive generation steps.</em></p>

</div>

**Example:**

```python
gen_tokens = model.generate(
    **input_ids,
    num_beams=3,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

**With:**

```text
num_beams = 3
```

the decoder maintains a beam of approximately 3 high-scoring candidate sequences during generation.

---

# 🧭 **How Beam Search Works**

**Suppose the current context is:**

```text
The
```

**Candidate next tokens:**

```text
dog   → 0.40
nice  → 0.30
cat   → 0.20
...
```

**With `num_beams=3`, we keep:**

```text
The dog
The nice
The cat
```

Then each candidate is expanded.

```text
The dog
 ├── has
 ├── is
 └── ...

The nice
 ├── woman
 ├── ...
 └── ...

The cat
 ├── is
 ├── ...
 └── ...
```

The resulting candidate sequences are scored and only the strongest beam candidates are retained.

```text
                 The
                  │
        ┌─────────┼─────────┐
        ↓         ↓         ↓
      dog       nice       cat
        │         │         │
       ...       ...       ...
         \         |         /
          \________|________/
                   ↓
            Keep top beams
                   ↓
                Continue
```

---

# 📐 **Probability of an Entire Sequence**

For a generated sequence

```text
x₁, x₂, ..., x_T
```

**the autoregressive probability is:**

$$
P(x_1,x_2,\ldots,x_T)
=
\prod_{t=1}^{T}
P(x_t \mid x_1,\ldots,x_{t-1})
$$

**Conditioning on the prompt \(c\):**

$$
P(x_{1:T}\mid c)
=
\prod_{t=1}^{T}
P(x_t\mid c,x_{<t})
$$

Because multiplying many small probabilities can become numerically unstable, sequence scoring is commonly expressed using **log-probabilities**:

$$
\log P(x_{1:T}\mid c)
=
\sum_{t=1}^{T}
\log P(x_t\mid c,x_{<t})
$$

Beam search uses cumulative sequence scores to compare candidate paths.

---

# ⚠️ **Length Bias in Sequence Scoring**

A subtle issue appears when comparing sequences of different lengths.

**Because probabilities are between 0 and 1:**

$$
0 < P(x_t\mid x_{<t}) \leq 1
$$

multiplying more probabilities generally makes the product smaller.

Therefore, raw sequence probability can favor shorter sequences.

A common family of approaches is **length normalization / length penalty**.

**A simple conceptual normalized score is:**

$$
\text{score}
=
\frac{\sum_{t=1}^{T}\log P(x_t\mid x_{<t})}
{T}
$$

> Exact beam-search scoring and length-penalty behavior depend on the generation implementation and configuration.

---

# 🆚 **Greedy Search vs Beam Search**

| Feature | Greedy Search | Beam Search |
|---|---|---|
| Active sequences | 1 | Multiple |
| Looks at alternatives | No | Yes |
| Typical randomness | No | No |
| Compute cost | Lower | Higher |
| Main idea | Best token now | Keep best candidate paths |
| `num_beams` | 1 | Greater than 1 |

**Remember:**

```text
Greedy:
Best next token

**Beam:**
Best few candidate sequences
```

---

# 🎲 **Sampling**

## **Sampling = Randomly draw from the probability distribution**

Instead of always choosing the maximum-probability token, sampling draws a token according to the model's probability distribution.

**Enable sampling in Hugging Face with:**

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

**Suppose:**

```text
dog   → 0.50
cat   → 0.30
car   → 0.15
bat   → 0.05
```

**Sampling means:**

```text
dog → 50% chance
cat → 30% chance
car → 15% chance
bat →  5% chance
```

So the same prompt can produce different outputs on different runs.

---

# 🎯 **Sampling Is Not Uniform Randomness**

**Incorrect idea:**

```text
Pick any vocabulary token with equal probability.
```

**Correct idea:**

```text
Sample according to the model's probability distribution.
```

A high-probability token is still more likely to be selected.

---

# 🔢 **Sampling 100 Times**

**Suppose the final distribution is:**

```text
Token 1 → 0.50
Token 2 → 0.30
Token 3 → 0.20
```

**The expected number of occurrences over 100 independent samples is:**

$$
E[N_1]=100\times0.50=50
$$

$$
E[N_2]=100\times0.30=30
$$

$$
E[N_3]=100\times0.20=20
$$

**So we may see approximately:**

```text
Token 1 → 50
Token 2 → 30
Token 3 → 20
```

but the counts will **not necessarily be exactly** 50, 30, and 20 in one experiment.

---

# 🎯 **Top-K Sampling**

## **Top-K Sampling = Keep only the K highest-probability tokens, then sample**

**Suppose:**

```text
The →

dog  → 0.40
nice → 0.25
car  → 0.15
bat  → 0.10
cat  → 0.06
hat  → 0.04
```

**Set:**

```text
top_k = 3
```

**Then keep:**

```text
dog
nice
car
```

and discard the lower-ranked candidates for that step.

---

# 🧭 **Top-K Workflow**

```text
Full vocabulary distribution
            ↓
       Sort by probability
            ↓
       Keep top K tokens
            ↓
      Remove the rest
            ↓
       Renormalize
            ↓
         Sample
```

**If the original probabilities are:**

```text
dog  = 0.40
nice = 0.25
car  = 0.15
```

**then the retained mass is:**

$$
0.40+0.25+0.15=0.80
$$

**The probabilities are renormalized:**

$$
P'(\text{dog})
=
\frac{0.40}{0.80}
=
0.50
$$

$$
P'(\text{nice})
=
\frac{0.25}{0.80}
=
0.3125
$$

$$
P'(\text{car})
=
\frac{0.15}{0.80}
=
0.1875
$$

So sampling occurs from the renormalized top-k distribution.

---

# 💻 **Top-K with Hugging Face**

```python
prompt = "Once upon a time"

input_ids = tokenizer(prompt, return_tensors="pt")

gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    top_k=3,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

**Repeated runs can produce different continuations because:**

```text
do_sample=True
```

---

# 🔢 **Why Top-K Reduces the Search Space**

**If:**

```text
Vocabulary size = 50,000
```

then unrestricted sampling considers the entire distribution.

**With:**

```text
top_k = 3
```

only the top 3 candidates are retained for that step.

**So:**

```text
50,000 possible tokens
        ↓
     top_k = 3
        ↓
3 candidate tokens
```

> **Top-K uses a fixed number of candidate tokens.**

---

# 🌊 **Top-P / Nucleus Sampling**

## **Top-P = Keep the smallest set of high-probability tokens whose cumulative probability reaches P**

**Top-K asks:**

> **How many tokens should we keep?**

**Top-P asks:**

> **How much probability mass should we keep?**

---

# 🧮 **Top-P Example**

**Suppose:**

```text
The →

dog  → 0.30
nice → 0.20
car  → 0.15
bat  → 0.15
cat  → 0.10
hat  → 0.10
```

**Set:**

```text
top_p = 0.70
```

**Sort in descending probability:**

```text
dog  → 0.30
nice → 0.20
car  → 0.15
bat  → 0.15
cat  → 0.10
hat  → 0.10
```

**Cumulative probabilities:**

$$
0.30
$$

$$
0.30+0.20=0.50
$$

$$
0.30+0.20+0.15=0.65
$$

$$
0.30+0.20+0.15+0.15=0.80
$$

**The smallest set that reaches \(0.70\) is:**

```text
dog
nice
car
bat
```

Then the retained probabilities are renormalized before sampling.

**The retained mass is:**

$$
0.80
$$

**Therefore:**

$$
P'(\text{dog})
=
\frac{0.30}{0.80}
=
0.375
$$

$$
P'(\text{nice})
=
\frac{0.20}{0.80}
=
0.25
$$

$$
P'(\text{car})
=
\frac{0.15}{0.80}
=
0.1875
$$

$$
P'(\text{bat})
=
\frac{0.15}{0.80}
=
0.1875
$$

---

# 🧭 **Top-P Workflow**

```text
Full probability distribution
            ↓
     Sort by probability
            ↓
   Compute cumulative mass
            ↓
Keep smallest set reaching P
            ↓
       Renormalize
            ↓
          Sample
```

---

# 💻 **Top-P with Hugging Face**

```python
prompt = "Once upon a time"

input_ids = tokenizer(prompt, return_tensors="pt")

gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    top_p=0.7,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

# 🆚 **Top-K vs Top-P**

| Feature | Top-K | Top-P |
|---|---|---|
| Main parameter | `k` | `p` |
| Selection rule | Fixed number of tokens | Fixed probability mass |
| Candidate count | Approximately fixed | Changes with distribution |
| Adapts to distribution shape | Less | More |
| Sampling required | Usually used with `do_sample=True` | Usually used with `do_sample=True` |

**Memory trick:**

```text
Top-K
→ K = Number of tokens

Top-P
→ P = Probability mass
```

---

# 🌡️ **Temperature**

## **Temperature controls the sharpness of the probability distribution**

**The temperature-scaled softmax is:**

$$
P_T(x_i)
=
\frac{\exp(z_i/T)}
{\sum_j \exp(z_j/T)}
$$

**where:**

```text
z_i = logit for token i
T   = temperature
```

Temperature changes the **relative concentration** of probability mass.

---

# 🔥 **Low Temperature**

**When:**

$$
0<T<1
$$

the distribution becomes **sharper**.

The largest logits become more dominant.

**Conceptually:**

```text
Low temperature
      ↓
Sharper distribution
      ↓
More probability on top tokens
      ↓
Less randomness
```

---

# 🌡️ **High Temperature**

**When:**

$$
T>1
$$

the distribution becomes **flatter**.

Lower-ranked candidates receive relatively more probability.

**Conceptually:**

```text
High temperature
      ↓
Flatter distribution
      ↓
More probability spread
      ↓
More randomness
```

> People often describe this as “more creative,” but technically temperature changes **randomness / concentration**, not creativity directly.

---

# 🧮 **Softmax from Logits**

**Suppose there are 3 classes:**

```text
logits = [2.0, 1.0, 0.1]
```

**The softmax formula is:**

$$
P_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

**So:**

$$
P_1
=
\frac{e^{2.0}}
{e^{2.0}+e^{1.0}+e^{0.1}}
\approx 0.65900114
$$

$$
P_2
=
\frac{e^{1.0}}
{e^{2.0}+e^{1.0}+e^{0.1}}
\approx 0.24243297
$$

$$
P_3
=
\frac{e^{0.1}}
{e^{2.0}+e^{1.0}+e^{0.1}}
\approx 0.09856589
$$

**Therefore:**

```text
Class 1 → 65.90%
Class 2 → 24.24%
Class 3 →  9.86%
```

---

<div align="center">

<img src="assets/softmax_.png" width="800" alt="Softmax transformation from logits to probabilities">

<p><em>Figure: Converting logits into probabilities with softmax.</em></p>

</div>

---

# 🧪 **Softmax with NumPy**

```python
import numpy as np

logits = np.array([2.0, 1.0, 0.1])

probs = np.exp(logits) / np.sum(np.exp(logits))

print(probs)
```

**Output:**

```text
array([0.65900114, 0.24243297, 0.09856589])
```

**A numerically safer implementation subtracts the maximum logit before exponentiation:**

```python
def softmax(logits):
    logits = np.asarray(logits, dtype=np.float64)

    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)

    return exp_values / np.sum(exp_values)
```

The subtraction does not change the resulting probabilities because softmax is invariant to adding or subtracting the same constant from every logit.

---

# 🌡️ **Temperature-Scaled Softmax**

```python
def softmax_with_temp(logits, temp):
    if temp <= 0:
        raise ValueError("Temperature must be > 0.")

    logits = np.asarray(logits, dtype=np.float64)

    scaled = logits / temp
    scaled = scaled - np.max(scaled)

    exp_values = np.exp(scaled)

    return exp_values / np.sum(exp_values)
```

---

# 🌡️ **Temperature = 0.1**

**For:**

```text
logits = [2.0, 1.0, 0.1]
```

**and:**

```text
T = 0.1
```

**we obtain approximately:**

$$
P
\approx
[0.999954597,\;0.000045398,\;0.0000000056]
$$

**or:**

```text
Class 1 → ~100%
Class 2 → ~0%
Class 3 → ~0%
```

The distribution becomes extremely concentrated.

---

# 🌡️ **Temperature = 1**

**For:**

```text
T = 1
```

**we get ordinary softmax:**

$$
P
\approx
[0.65900114,\;0.24243297,\;0.09856589]
$$

---

# 🌡️ **Temperature = 2**

**For:**

```text
T = 2
```

**we obtain approximately:**

$$
P
\approx
[0.50168776,\;0.30428901,\;0.19402324]
$$

The distribution is much flatter.

---

# 🧠 **How Temperature Works**

**Suppose:**

```text
logits = [2.0, 1.0, 0.1]
```

**Before temperature:**

```text
2.0
1.0
0.1
```

**Divide by a small temperature:**

```text
T = 0.1

20
10
1
```

The differences become much larger, so softmax becomes very sharp.

**Divide by a large temperature:**

```text
T = 2

1.0
0.5
0.05
```

The differences become smaller, so softmax becomes flatter.

```text
Low T
→ amplify relative differences

High T
→ reduce relative differences
```

---

# ⚠️ **Temperature = 0**

The expression

$$
\frac{z_i}{T}
$$

**is undefined at exactly:**

$$
T=0
$$

**The relevant limiting behavior is:**

$$
\lim_{T\to0^+}
P_T(x_i)
$$

which concentrates probability on the maximum-logit token.

**So, conceptually:**

```text
T → 0+
        ↓
argmax-like behavior
```

---

# 💻 **Temperature with Hugging Face**

```python
prompt = "Once upon a time"

input_ids = tokenizer(prompt, return_tensors="pt")

gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    temperature=0.9,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

**When:**

```python
do_sample=True
```

the temperature affects the sampling distribution.

---

# 🎛️ **Temperature Comparison**

```text
             Probability distribution
                        │
          ┌─────────────┴─────────────┐
          ↓                           ↓
      Low T                         High T
          ↓                           ↓
      Sharper                       Flatter
          ↓                           ↓
   More focused                  More spread
          ↓                           ↓
    Less random                  More random
```

**A useful memory trick:**

```text
Low temperature:
"Prefer the strongest choices."

High temperature:
"Explore more of the distribution."
```

---

# 🧠 **Logits vs Probabilities**

This distinction is extremely important.

## **Logits**

**Logits are raw model scores:**

```text
token A → 2.0
token B → 1.0
token C → 0.1
```

They do not need to sum to 1.

## **Probabilities**

**After softmax:**

```text
token A → 0.659
token B → 0.242
token C → 0.099
```

**Now:**

$$
\sum_i P_i=1
$$

****So:****

```text
Logits
  ↓
Temperature scaling
  ↓
Softmax
  ↓
Probabilities
  ↓
Filtering / sampling / selection
```

---

# 🧩 ****What Exactly Is a Decoding Strategy?****

****At every generation step, the model gives:****

```text
Vocab token → score/probability
```

****Decoding answers:****

> **Given this distribution, which token should be generated next?**

Different methods answer that question differently.

```text
Greedy
→ choose highest probability

Beam Search
→ keep multiple high-scoring paths

Sampling
→ draw according to probability

Top-K
→ sample only from K strongest candidates

Top-P
→ sample only from a probability-mass nucleus

Temperature
→ reshape the distribution before sampling
```

---

# 🛠️ ****Combining Generation Controls****

****A practical generation call may combine several settings:****

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    max_new_tokens=100,
    no_repeat_ngram_size=3
)

print(tokenizer.decode(gen_tokens[0]))
```

****Conceptually:****

```text
Model
  ↓
Logits
  ↓
Temperature scaling
  ↓
Top-K / Top-P filtering
  ↓
Renormalize
  ↓
Sampling
  ↓
Next token
  ↓
Append token
  ↓
Repeat
```

> **Do not assume that every parameter should always be enabled.** Generation settings should match the task.

---

# ⚙️ **`do_sample`**

This parameter determines whether stochastic sampling is used.

```python
do_sample=False
```

typically corresponds to deterministic selection behavior such as greedy decoding when no beam search is requested.

```python
do_sample=True
```

enables sampling-based generation.

**A useful comparison:**

```text
do_sample=False
→ deterministic-style selection

do_sample=True
→ stochastic sampling
```

The exact generation behavior can also depend on other configuration parameters.

---

# 🔢 **`num_beams`**

Controls beam search width.

```python
num_beams=1
```

does not maintain multiple beams.

```python
num_beams=3
```

maintains multiple candidate paths.

Larger values can increase search breadth and computation.

```text
num_beams ↑
    ↓
More candidate sequences
    ↓
More computation
```

---

# 🛑 **When Does Generation Stop?**

Generation can stop for several reasons.

```text
1. EOS / end-of-sequence token is generated
2. max_new_tokens is reached
3. A custom stopping criterion is triggered
```

**Conceptually:**

```text
Generate token
      ↓
Is it an EOS token?
   ┌──┴──┐
  Yes    No
   ↓      ↓
 Stop   Continue
```

---

# 🧠 **Autoregressive Generation**

**The complete language-model factorization is:**

$$
P(x_1,x_2,\ldots,x_T)
=
\prod_{t=1}^{T}
P(x_t\mid x_1,\ldots,x_{t-1})
$$

**For a prompt \(c\):**

$$
P(y_{1:T}\mid c)
=
\prod_{t=1}^{T}
P(y_t\mid c,y_{<t})
$$

This explains why generation is sequential.

```text
Token 1
   ↓
Token 2
   ↓
Token 3
   ↓
Token 4
   ↓
...
```

Each new token changes the context for the next prediction.

---

# 🔄 **One Full Generation Step**

**Suppose:**

```text
Prompt:
"Once upon a time"
```

### **Step 1 — Tokenize**

```text
Text
 ↓
Tokenizer
 ↓
Input IDs
```

### **Step 2 — Model forward pass**

```text
Input IDs
 ↓
Decoder-only Transformer
 ↓
Logits
```

### **Step 3 — Convert scores to a distribution**

```text
Logits
 ↓
Temperature / other processing
 ↓
Probabilities
```

### **Step 4 — Decode**

```text
Probabilities
 ↓
Greedy / Beam / Sampling
 ↓
Next token
```

### **Step 5 — Append**

```text
Prompt + next token
```

### **Step 6 — Repeat**

```text
Prompt + token 1
       ↓
Predict token 2
       ↓
Predict token 3
       ↓
...
```

---

# 🌳 **Greedy Search Example**

```text
Prompt:
"The"
```

**Suppose:**

```text
dog    0.50
nice   0.30
cat    0.20
```

**Greedy selects:**

```text
dog
```

**Now:**

```text
"The dog"
```

**Suppose the next distribution is:**

```text
has    0.60
is     0.25
runs   0.15
```

**Greedy selects:**

```text
has
```

**Now:**

```text
"The dog has"
```

The process repeats.

---

# 🌳 **Beam Search Example**

**Suppose:**

```text
Prompt:
"The"
```

**and:**

```text
dog    0.50
nice   0.30
cat    0.20
```

**For:**

```text
num_beams = 2
```

**keep:**

```text
"The dog"
"The nice"
```

**Now expand both:**

```text
"The dog"  → has, is, runs, ...
"The nice" → woman, day, ...,
```

Score the expanded candidates and keep the strongest 2.

```text
Two best active sequences
          ↓
Expand
          ↓
Score
          ↓
Keep two best
          ↓
Repeat
```

---

# 🎲 **Sampling Example**

**Suppose:**

```text
dog    0.50
nice   0.25
cat    0.15
car    0.10
```

**One sample may choose:**

```text
dog
```

**Another may choose:**

```text
nice
```

**Another may choose:**

```text
dog
```

**Another may choose:**

```text
cat
```

Because selection is probabilistic, repeated runs can differ.

---

# 📊 **Randomness vs Determinism**

```text
                    Generation
                        │
            ┌───────────┴───────────┐
            ↓                       ↓
       Deterministic            Stochastic
            ↓                       ↓
        Greedy                  Sampling
        Beam Search              Top-K
                                  Top-P
                               Temperature
```

**A useful rule:**

```text
No sampling
→ usually repeatable

Sampling
→ can vary across runs
```

---

# ⚠️ **Important Correction: Temperature Is Usually Used with Sampling**

Temperature changes the probability distribution.

**If the decoding procedure simply chooses:**

$$
\operatorname{argmax}_i z_i
$$

**then any positive temperature preserves the ordering of logits:**

$$
\operatorname{argmax}_i \frac{z_i}{T}
=
\operatorname{argmax}_i z_i
\qquad
\text{for }T>0
$$

Therefore, temperature has its main practical effect when the system **samples from the distribution**.

**This is why a typical Hugging Face configuration uses:**

```python
do_sample=True,
temperature=0.8
```

rather than relying on temperature alone.

---

# 🧠 **Top-K and Top-P Are Filtering Methods**

**A useful distinction:**

```text
Temperature
→ Reweights / reshapes probabilities

Top-K
→ Filters candidate tokens by rank

Top-P
→ Filters candidate tokens by cumulative probability

Sampling
→ Randomly selects from the resulting distribution
```

**So a common pipeline is:**

```text
Logits
 ↓
Temperature
 ↓
Top-K / Top-P filtering
 ↓
Renormalization
 ↓
Sampling
```

---

# 📐 **Mathematical View of Top-K**

Let \(V\) be the vocabulary and let \(K\) be the desired number of candidates.

**Let:**

$$
S_K = \text{the set of the }K\text{ highest-probability tokens}
$$

**Then the filtered distribution is:**

$$
P_K(v)
=
\begin{cases}
\displaystyle
\frac{P(v)}{\sum_{u\in S_K}P(u)}
& v\in S_K \\[12pt]
0
& v\notin S_K
\end{cases}
$$

Then sample from \(P_K\).

---

# 📐 **Mathematical View of Top-P**

**Let the tokens be sorted so:**

$$
P(v_1)\ge P(v_2)\ge\cdots\ge P(v_{|V|})
$$

**Choose the smallest \(m\) such that:**

$$
\sum_{i=1}^{m}P(v_i)\ge p
$$

**The nucleus set is:**

$$
S_p=\{v_1,\ldots,v_m\}
$$

**The filtered distribution becomes:**

$$
P_p(v)
=
\begin{cases}
\displaystyle
\frac{P(v)}{\sum_{u\in S_p}P(u)}
& v\in S_p \\[12pt]
0
& v\notin S_p
\end{cases}
$$

Then sample from \(P_p\).

---

# 🧮 **Mathematical View of Temperature**

**Given logits:**

$$
z_1,z_2,\ldots,z_{|V|}
$$

**temperature \(T\) produces:**

$$
\tilde z_i=\frac{z_i}{T}
$$

**Then:**

$$
P_T(i)
=
\frac{e^{\tilde z_i}}
{\sum_j e^{\tilde z_j}}
=
\frac{e^{z_i/T}}
{\sum_j e^{z_j/T}}
$$

**Thus:**

```text
small T → sharper distribution
large T → flatter distribution
```

---

# 🧠 **Greedy vs Sampling in One Example**

**Suppose:**

```text
dog  → 0.55
cat  → 0.30
car  → 0.10
bat  → 0.05
```

## **Greedy**

**Always:**

```text
dog
```

**because:**

$$
\operatorname{argmax}(0.55,0.30,0.10,0.05)=\text{dog}
$$

## **Sampling**

```text
dog → 55%
cat → 30%
car → 10%
bat →  5%
```

So different runs can choose different tokens.

---

# 🧠 **Why Probability Mass Matters**

**Suppose:**

```text
Token A → 0.85
Token B → 0.05
Token C → 0.03
Token D → 0.02
Token E → 0.01
Token F → 0.01
Token G → 0.01
Token H → 0.01
Token I → 0.00...
```

A top-k strategy might arbitrarily keep a fixed count.

**A top-p strategy can adapt:**

```text
Keep A
and only enough other tokens
to reach the chosen probability mass.
```

This is the central motivation for **nucleus sampling**.

---

# 🆚 **Decoding Strategy Summary**

| Strategy / Control | Main Idea | Deterministic? | Main Use |
|---|---|---:|---|
| Greedy | Choose highest-probability next token | Yes | Simple, predictable generation |
| Beam Search | Keep multiple high-scoring sequences | Usually yes | Search over candidate paths |
| Sampling | Draw from distribution | No | Diverse generation |
| Top-K | Sample from K highest-probability tokens | No | Restrict low-probability tail |
| Top-P | Sample from cumulative probability nucleus | No | Adaptive filtering |
| Temperature | Change distribution sharpness | Depends on decoder | Control randomness |
| `no_repeat_ngram_size` | Block repeated n-grams | Depends on decoder | Reduce repetition |
| `max_new_tokens` | Limit newly generated tokens | N/A | Control output length |

---

# 🎯 **Practical Intuition**

## **Greedy Search**

```text
"I want the safest next token."
```

## **Beam Search**

```text
"I want to explore several strong paths."
```

## **Sampling**

```text
"I want a probabilistic choice."
```

## **Top-K**

```text
"I only want the K strongest candidates."
```

## **Top-P**

```text
"I want enough candidates to cover P probability mass."
```

## **Temperature**

```text
"I want to control how concentrated the distribution is."
```

---

# 🧪 **A Clean Hugging Face Comparison**

## **Greedy**

```python
gen_tokens = model.generate(
    **input_ids,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

## **Beam Search**

```python
gen_tokens = model.generate(
    **input_ids,
    num_beams=3,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

## **Sampling**

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

## **Top-K Sampling**

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    top_k=3,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

## **Top-P Sampling**

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    top_p=0.7,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

## **Temperature Sampling**

```python
gen_tokens = model.generate(
    **input_ids,
    do_sample=True,
    temperature=0.9,
    max_new_tokens=50
)

print(tokenizer.decode(gen_tokens[0]))
```

---

# ⚠️ **Common Mistakes**

## **1. Logits are not probabilities**

**Wrong:**

```text
logits = probabilities
```

**Correct:**

```text
logits
  ↓
softmax
  ↓
probabilities
```

---

## **2. Sampling does not mean uniform randomness**

**Wrong:**

```text
Every token gets equal probability.
```

**Correct:**

```text
Tokens are sampled according to their probabilities.
```

---

## **3. Top-K and Top-P are not the same**

```text
Top-K → fixed candidate count

Top-P → probability-mass-based candidate set
```

---

## **4. High temperature does not guarantee nonsense**

Temperature changes the distribution.

It does not guarantee that the output will be bad.

**Output quality also depends on:**

```text
Model
+
Prompt
+
Temperature
+
Top-K / Top-P
+
Other generation settings
```

---

## **5. Beam search is not sampling**

```text
Beam Search
→ search among multiple candidate paths

Sampling
→ random draw from a probability distribution
```

---

## **6. `max_new_tokens` is not the same as total sequence length**

```text
max_new_tokens
→ controls newly generated tokens

max_length
→ controls total sequence length in the relevant generation setup
```

---

## **7. Temperature is not a synonym for creativity**

**A better technical statement is:**

> **Temperature controls the concentration of the sampling distribution.**

Lower temperature tends to make outputs more focused; higher temperature tends to increase stochastic variation.

---

# 🧠 **End-to-End Generation Pipeline**

```text
                         Prompt
                           ↓
                       Tokenizer
                           ↓
                        Input IDs
                           ↓
                Decoder-only Transformer
                           ↓
                          Logits
                           ↓
                  Temperature Scaling
                           ↓
                  Top-K / Top-P Filtering
                           ↓
                     Renormalization
                           ↓
                 Greedy / Beam / Sampling
                           ↓
                       Next Token
                           ↓
                  Append to Context
                           ↓
                 Check Stop Condition
                      ↙          ↘
                    Stop       Continue
                               ↓
                         Run Model Again
                               ↓
                             Repeat
```

---

# 📚 **Key Formulae**

## **Autoregressive factorization**

$$
P(x_{1:T})
=
\prod_{t=1}^{T}
P(x_t\mid x_{<t})
$$

---

## **Conditional generation from a prompt**

$$
P(y_{1:T}\mid c)
=
\prod_{t=1}^{T}
P(y_t\mid c,y_{<t})
$$

---

## **Greedy Search**

$$
y_t
=
\underset{v\in V}{\operatorname{argmax}}
\;P(v\mid c,y_{<t})
$$

---

## **Sequence probability**

$$
P(y_{1:T}\mid c)
=
\prod_{t=1}^{T}
P(y_t\mid c,y_{<t})
$$

---

## **Sequence log-probability**

$$
\log P(y_{1:T}\mid c)
=
\sum_{t=1}^{T}
\log P(y_t\mid c,y_{<t})
$$

---

## **Softmax**

$$
P_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}
$$

---

## **Temperature-scaled softmax**

$$
P_T(i)
=
\frac{e^{z_i/T}}
{\sum_j e^{z_j/T}}
$$

---

## **Top-K filtering**

$$
P_K(v)
=
\begin{cases}
\displaystyle
\frac{P(v)}
{\sum_{u\in S_K}P(u)}
& v\in S_K \\[12pt]
0
& v\notin S_K
\end{cases}
$$

---

## **Top-P / Nucleus filtering**

$$
\sum_{i=1}^{m}P(v_i)\ge p
$$

**with:**

$$
P(v_1)\ge P(v_2)\ge\cdots
$$

**and the nucleus:**

$$
S_p=\{v_1,\ldots,v_m\}
$$

followed by renormalization.

---

# 🧭 **Recommended Learning Order**

```text
1. Decoder-only Transformer
        ↓
2. Autoregressive generation
        ↓
3. Vocabulary and next-token prediction
        ↓
4. Logits
        ↓
5. Softmax
        ↓
6. Greedy Search
        ↓
7. Beam Search
        ↓
8. Sampling
        ↓
9. Top-K
        ↓
10. Top-P / Nucleus Sampling
        ↓
11. Temperature
        ↓
12. Repetition constraints
        ↓
13. Stopping criteria
        ↓
14. Combining generation controls
```

---

# 📝 **Important Hugging Face Generation Parameters**

```text
max_new_tokens
→ Maximum number of newly generated tokens

do_sample
→ Enable stochastic sampling

num_beams
→ Number of beam candidates

top_k
→ Restrict sampling to the top K tokens

top_p
→ Restrict sampling to a cumulative probability nucleus

temperature
→ Control the sharpness of the sampling distribution

no_repeat_ngram_size
→ Prevent repeated n-grams
```

---

# 🔗 **Generation Configuration Examples**

## **Deterministic generation**

```python
generation_kwargs = {
    "max_new_tokens": 100
}

gen_tokens = model.generate(
    **input_ids,
    **generation_kwargs
)
```

---

## **Beam search**

```python
generation_kwargs = {
    "num_beams": 4,
    "max_new_tokens": 100
}

gen_tokens = model.generate(
    **input_ids,
    **generation_kwargs
)
```

---

## **Sampling with top-k**

```python
generation_kwargs = {
    "do_sample": True,
    "top_k": 50,
    "max_new_tokens": 100
}

gen_tokens = model.generate(
    **input_ids,
    **generation_kwargs
)
```

---

## **Sampling with top-p and temperature**

```python
generation_kwargs = {
    "do_sample": True,
    "top_p": 0.95,
    "temperature": 0.8,
    "max_new_tokens": 100
}

gen_tokens = model.generate(
    **input_ids,
    **generation_kwargs
)
```

---

# 📌 **Quick Revision**

```text
Decoder-only LM
→ Predicts the next token autoregressively

Logits
→ Raw scores produced by the model

Softmax
→ Converts logits into probabilities

Greedy Search
→ Always choose the maximum-probability token

Beam Search
→ Keep multiple high-scoring candidate sequences

Sampling
→ Randomly sample from a probability distribution

Top-K
→ Keep K highest-probability candidates

Top-P
→ Keep the smallest set reaching cumulative probability P

Temperature
→ Control distribution sharpness

no_repeat_ngram_size
→ Prevent repeated n-grams

max_new_tokens
→ Limit the amount of new text
```

---

# 🧠 **Final Mental Model**

```text
                    Decoder-Only LM
                           │
                           ↓
                Predict Next-Token Logits
                           │
                           ↓
                      Probability
                       Distribution
                           │
             ┌─────────────┼─────────────┐
             ↓             ↓             ↓
          Greedy        Beam Search    Sampling
             │             │             │
             │             │        ┌────┼────┐
             │             │        ↓    ↓    ↓
             │             │      Top-K Top-P Temp.
             │             │        │    │    │
             └─────────────┴────────┴────┴────┘
                           ↓
                      Next Token
                           ↓
                  Append to Context
                           ↓
                       Repeat
                           ↓
                    Generated Text
```

> **Core Idea:** The language model produces a probability distribution over possible next tokens. **Decoding strategies determine how that distribution is converted into an actual sequence of tokens.**

---

# 🎓 **One-Line Definition**

> **Decoding is the inference-time process of selecting the next token from a language model's predicted distribution, using methods such as greedy search, beam search, or sampling techniques such as top-k, top-p, and temperature-controlled sampling.**
